# 02 — Chargement de la base SIRENE

Charge depuis les parquets INSEE :
- **Etab actifs** (vue `etab_active`) : pour la siretisation
- **UL + adresse siège** (vue `etab_siege`) : pour la sirenisation

Sortie : 2 fichiers parquet dans `data/interim/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.connexion   import get_duckdb_connection
from src.display     import afficher_tableau
from config.settings import (
    SIRENE_ETAB_RAW, SIRENE_UL_RAW, INTERIM_DIR,
)

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Établissements actifs (siretisation)

Vue `etab_active` : tous les établissements actifs avec dénomination UL et enseignes établissement.

In [2]:
duckdb_con = get_duckdb_connection()

df_etab = duckdb_con.execute("""
    SELECT siren, siret, nic,
           denominationUniteLegale, sigleUniteLegale,
           categorieJuridiqueUniteLegale, activitePrincipaleUniteLegale,
           dateCreationUniteLegale,
           etablissementSiege,
           dateCreationEtablissement,
           enseigne1Etablissement, enseigne2Etablissement,
           enseigne3Etablissement, denominationUsuelleEtablissement,
           numeroVoieEtablissement, typeVoieEtablissement,
           libelleVoieEtablissement, codeCommuneEtablissement
    FROM etab_active
""").df()

print(f'Établissements SIRENE actifs : {len(df_etab):,}')

df_etab.to_parquet(SIRENE_ETAB_RAW, index=False)
print(f'Sauvegardé : {SIRENE_ETAB_RAW}')

afficher_tableau(df_etab, 'Aperçu Etab SIRENE')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Établissements SIRENE actifs : 16,867,946
Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/interim/sirene_etab.parquet


siren,siret,nic,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,dateCreationUniteLegale,etablissementSiege,dateCreationEtablissement,enseigne1Etablissement,enseigne2Etablissement,enseigne3Etablissement,denominationUsuelleEtablissement,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement
000325175,00032517500065,00065,None,None,1000,32.12Z,2000-09-26 00:00:00,True,2018-02-07 00:00:00,None,None,None,None,51,RUE,MARX DORMOY,13204
005420021,00542002100056,00056,ETABLISSEMENTS LUCIEN BIQUEZ,None,5710,46.69B,1954-01-01 00:00:00,True,2009-12-23 00:00:00,None,None,None,None,21,BOULEVARD,DES PRES,80001
005420120,00542012000015,00015,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01 00:00:00,False,1989-01-27 00:00:00,None,None,None,None,None,RUE,DE LA FONTAINE,80688
005420120,00542012000023,00023,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01 00:00:00,False,1900-01-01 00:00:00,None,None,None,None,12,ROUTE,DE MONTREUIL,62688
005420120,00542012000056,00056,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01 00:00:00,True,2026-01-13 00:00:00,None,None,None,None,32,CHEMIN,DES GARENNES,80713


## 2. UL avec adresse siège (sirenisation)

Vue `etab_siege` : une ligne par UL active, avec l'adresse de son siège.

In [3]:
df_ul = duckdb_con.execute("""
    SELECT siren, denominationUniteLegale, sigleUniteLegale,
           categorieJuridiqueUniteLegale, activitePrincipaleUniteLegale,
           dateCreationUniteLegale,
           nicSiegeUniteLegale,
           numeroVoieEtablissement, typeVoieEtablissement,
           libelleVoieEtablissement, codeCommuneEtablissement
    FROM etab_siege
""").df()
duckdb_con.close()

print(f'UL SIRENE actives : {len(df_ul):,}')
print(f'  avec dénomination : {df_ul["denominationUniteLegale"].notna().sum():,}')
print(f'  avec sigle        : {df_ul["sigleUniteLegale"].notna().sum():,}')
print(f'  avec APE          : {df_ul["activitePrincipaleUniteLegale"].notna().sum():,}')
print(f'  avec date création: {df_ul["dateCreationUniteLegale"].notna().sum():,}')

df_ul.to_parquet(SIRENE_UL_RAW, index=False)
print(f'Sauvegardé : {SIRENE_UL_RAW}')

afficher_tableau(df_ul, 'Aperçu UL SIRENE')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



UL SIRENE actives : 15,184,224
  avec dénomination : 10,036,225
  avec sigle        : 2,594,960
  avec APE          : 15,184,224
  avec date création: 15,184,224
Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/interim/sirene_ul.parquet


siren,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,dateCreationUniteLegale,nicSiegeUniteLegale,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement
056801558,COMPAGNIE FRANCAISE DES NAPHTES,None,5499,66.30Z,1956-01-01 00:00:00,00034,24,AVENUE,DE LA MER,83112
056801608,SOC FRANC PROD TARTRIQUE MANTE,None,5710,64.30Z,1956-01-01 00:00:00,00045,302,RUE,GARIBALDI,69387
056801715,VOLUMAIR INTERNATIONALE,None,5710,46.69B,1956-01-01 00:00:00,00022,None,None,N 96,13005
056801848,SOCIETE DE RESTAURATION DES GOUDES,None,5499,68.20B,1956-01-01 00:00:00,00013,None,RUE,DESIRE PELAPRAT,13208
056802093,ANCIENS ETS MICHEL ET CIE SA,None,5499,46.18Z,1956-01-01 00:00:00,00031,25,BOULEVARD,MASSENET,13214
